Conexión base de datos

In [30]:
# Import libraries
import sqlite3
import pandas as pd

# Connect to the database
conn = sqlite3.connect('toys_and_models.sqlite')
cursor = conn.cursor()

Exploración de tablas

In [31]:
# List of all tables
cursor.execute("SELECT name FROM sqlite_master WHERE type='table';")
tables = cursor.fetchall()

print("Table Dimensions and Column Information:")
print("-" * 50)

# Iterate over each table
for table_name in tables:
    current_table = table_name[0]
    
    # Number of rows for the current table
    cursor.execute(f"SELECT COUNT(*) FROM {current_table};")
    num_rows = cursor.fetchone()[0]

    # Get the number of columns and column information
    cursor.execute(f"PRAGMA table_info({current_table});")
    columns_info = cursor.fetchall()
    num_cols = len(columns_info)
    
    # Print the table name and its dimensions
    print(f"Table: {current_table}")
    print(f"  - Dimension: {num_rows} rows x {num_cols} columns")
    print("  - Columns:")
    
    # Print the column details
    for column in columns_info:
        print(f"    - {column[1]} (Type: {column[2]})")
    
    print("\n")

Table Dimensions and Column Information:
--------------------------------------------------
Table: customers
  - Dimension: 122 rows x 13 columns
  - Columns:
    - customerNumber (Type: int(11))
    - customerName (Type: varchar(50))
    - contactLastName (Type: varchar(50))
    - contactFirstName (Type: varchar(50))
    - phone (Type: varchar(50))
    - addressLine1 (Type: varchar(50))
    - addressLine2 (Type: varchar(50))
    - city (Type: varchar(50))
    - state (Type: varchar(50))
    - postalCode (Type: varchar(15))
    - country (Type: varchar(50))
    - salesRepEmployeeNumber (Type: int(11))
    - creditLimit (Type: decimal(10,2))


Table: employees
  - Dimension: 23 rows x 8 columns
  - Columns:
    - employeeNumber (Type: int(11))
    - lastName (Type: varchar(50))
    - firstName (Type: varchar(50))
    - extension (Type: varchar(10))
    - email (Type: varchar(100))
    - officeCode (Type: varchar(10))
    - reportsTo (Type: int(11))
    - jobTitle (Type: varchar(50))


T

General Overview

In [ ]:
print("General Overview:")
print("-" * 50)

# Total number of customers
cursor.execute("SELECT COUNT(*) FROM customers;")
customer_count = cursor.fetchone()[0]
print(f"Total Customers: {customer_count}")

# Total number of products
cursor.execute("SELECT COUNT(*) FROM products;")
product_count = cursor.fetchone()[0]
print(f"Total Products: {product_count}")

# Total number of employees
cursor.execute("SELECT COUNT(*) FROM employees;")
employee_count = cursor.fetchone()[0]
print(f"Total Employees: {employee_count}")

General Overview:
--------------------------------------------------
Total Customers: 122
Total Products: 110
Total Employees: 23


Credit Limit Analysis

In [ ]:
print("Credit Limit Analysis:")
print("-" * 50)

# Highest credit limit
cursor.execute("SELECT MAX(creditLimit) FROM customers;")
max_credit = cursor.fetchone()[0]
print(f"Highest Credit Limit: ${max_credit:,.2f}")

# Lowest credit limit
cursor.execute("SELECT MIN(creditLimit) FROM customers;")
min_credit = cursor.fetchone()[0]
print(f"Lowest Credit Limit: ${min_credit:,.2f}")

# Average credit limit
cursor.execute("SELECT AVG(creditLimit) FROM customers;")
avg_credit = cursor.fetchone()[0]
print(f"Average Credit Limit: ${avg_credit:,.2f}")

Credit Limit Analysis:
--------------------------------------------------
Highest Credit Limit: $227,600.00
Lowest Credit Limit: $0.00
Average Credit Limit: $67,659.02


Análisis de Órdenes con más de 500 Artículos

In [ ]:
print("Orders with more than 500 items:")
print("-" * 50)

# Query to Obtain orders with more than 500 items
query = """
    SELECT
        orderNumber,
        SUM(quantityOrdered) AS totalItems
    FROM
        orderdetails
    GROUP BY
        orderNumber
    HAVING
        SUM(quantityOrdered) > 500
    ORDER BY
        totalItems DESC;
"""

df_result = pd.read_sql(query, conn)
print(df_result)

Análisis de Ventas Totales por País:

In [ ]:
print("Total sales per country:")
print("-" * 50)

# Query to calculate total sales per country
query_sales_by_country = """
    SELECT
        c.country,
        SUM(od.quantityOrdered * od.priceEach) AS totalSales
    FROM
        customers c
    JOIN
        orders o ON c.customerNumber = o.customerNumber
    JOIN
        orderdetails od ON o.orderNumber = od.orderNumber
    GROUP BY
        c.country
    ORDER BY
        totalSales DESC;
"""

df_sales_by_country = pd.read_sql(query_sales_by_country, conn)
print(df_sales_by_country)

Análisis de Ventas Totales por País:
--------------------------------------------------
        country  totalSales
0           USA  2917020.76
1         Spain   892685.17
2        France   841384.63
3     Australia   479537.30
4            UK   399688.50
5         Italy   325254.55
6   New Zealand   311503.35
7       Finland   295149.35
8     Singapore   258155.56
9       Denmark   197356.30
10      Germany   196470.99
11       Canada   176841.48
12        Japan   167294.50
13     Norway     166621.51
14       Sweden   159671.81
15      Austria   127312.87
16  Switzerland   108777.92
17       Norway   104224.79
18      Belgium    91471.03
19  Philippines    87468.30
20      Ireland    49898.27
21    Hong Kong    45480.79


Relación entre Clientes y Empleados

In [ ]:
print("Customer and their sales representatives:")
print("-" * 50)

# Query to retrieve customer names and their sales representatives' names
query_customer_sales_rep = """
    SELECT
        c.customerName,
        e.firstName || ' ' || e.lastName AS salesRepName
    FROM
        customers c
    JOIN
        employees e ON c.salesRepEmployeeNumber = e.employeeNumber
    ORDER BY
        salesRepName, c.customerName;
"""

df_customer_sales_rep = pd.read_sql(query_customer_sales_rep, conn)
print(df_customer_sales_rep)

Relación de Clientes y Representantes de Ventas:
--------------------------------------------------
                    customerName     salesRepName
0         Annas Decorations, Ltd      Andy Fixter
1   Australian Collectables, Ltd      Andy Fixter
2     Australian Collectors, Co.      Andy Fixter
3    Australian Gift Network, Co      Andy Fixter
4      Souveniers And Things Co.      Andy Fixter
..                           ...              ...
95         Diecast Classics Inc.  Steve Patterson
96              FunGiftIdeas.com  Steve Patterson
97             Gifts4AllAges.com  Steve Patterson
98           Martas Replicas Co.  Steve Patterson
99  Online Diecast Creations Co.  Steve Patterson

[100 rows x 2 columns]


Análisis del Tamaño de los Pedidos

In [ ]:
print("Unique products in each order:")
print("-" * 50)

# Query to get the number of unique products in each order
query_order_size = """
    SELECT
        orderNumber,
        COUNT(productCode) AS uniqueProductsCount
    FROM
        orderdetails
    GROUP BY
        orderNumber
    ORDER BY
        uniqueProductsCount DESC;
"""

df_order_size = pd.read_sql(query_order_size, conn)
print(df_order_size)

Análisis del Tamaño de los Pedidos:
--------------------------------------------------
     orderNumber  uniqueProductsCount
0          10106                   18
1          10159                   18
2          10165                   18
3          10168                   18
4          10222                   18
..           ...                  ...
278        10294                    1
279        10317                    1
280        10345                    1
281        10364                    1
282        10376                    1

[283 rows x 2 columns]


Análisis de Relaciones

In [ ]:
print("Análisis de Relaciones con Diferentes Tipos de JOINs:")
print("-" * 50)

# -----  INNER JOIN -----
# Muestra solo a los clientes que tienen un representante de ventas
print("--- Clientes con un representante de ventas asignado (INNER JOIN) ---")
query_inner_join = """
    SELECT
        c.customerName,
        e.firstName || ' ' || e.lastName AS salesRepName
    FROM
        customers c
    JOIN
        employees e ON c.salesRepEmployeeNumber = e.employeeNumber;
"""
df_inner = pd.read_sql(query_inner_join, conn)
print(df_inner)
print("\n" + "="*80 + "\n")

# -----LEFT JOIN -----
# Muestra TODOS los clientes, incluso si no tienen un representante de ventas
print("--- Todos los clientes, incluyendo los que no tienen representante (LEFT JOIN) ---")
query_left_join = """
    SELECT
        c.customerName,
        e.firstName || ' ' || e.lastName AS salesRepName
    FROM
        customers c
    LEFT JOIN
        employees e ON c.salesRepEmployeeNumber = e.employeeNumber;
"""
df_left = pd.read_sql(query_left_join, conn)
print(df_left)
print("\n" + "="*80 + "\n")

# -----SELF-JOIN -----
# Muestra la jerarquía de los empleados: quién reporta a quién
print("--- Jerarquía de empleados (SELF-JOIN) ---")
query_self_join = """
    SELECT
        e.firstName || ' ' || e.lastName AS employeeName,
        m.firstName || ' ' || m.lastName AS managerName
    FROM
        employees e
    JOIN
        employees m ON e.reportsTo = m.employeeNumber
    ORDER BY
        managerName;
"""
df_self = pd.read_sql(query_self_join, conn)
print(df_self)
print("\n" + "="*80 + "\n")


Análisis de Relaciones con Diferentes Tipos de JOINs:
--------------------------------------------------
--- Clientes con un representante de ventas asignado (INNER JOIN) ---
                      customerName      salesRepName
0                Atelier graphique  Gerard Hernandez
1               Signal Gift Stores   Leslie Thompson
2       Australian Collectors, Co.       Andy Fixter
3                La Rochelle Gifts  Gerard Hernandez
4               Baane Mini Imports       Barry Jones
..                             ...               ...
95    Motor Mint Distributors Inc.     George Vanauf
96        Signal Collectibles Ltd.   Leslie Jennings
97  Double Decker Gift Stores, Ltd        Larry Bott
98            Diecast Collectables    Julie Firrelli
99                Kellys Gift Shop       Peter Marsh

[100 rows x 2 columns]


--- Todos los clientes, incluyendo los que no tienen representante (LEFT JOIN) ---
                       customerName      salesRepName
0                 Atelier 

Clasificación de Productos por Ventas Totales

In [ ]:
print("Rank products:")
print("-" * 50)

# Query to rank products by total sales
query_product_rank = """
    WITH ProductSales AS (
        SELECT
            p.productName,
            SUM(od.quantityOrdered * od.priceEach) AS totalSales
        FROM
            products p
        JOIN
            orderdetails od ON p.productCode = od.productCode
        GROUP BY
            p.productName
    )
    SELECT
        productName,
        totalSales,
        -- RANK() assigns a rank to each product based on total sales
        RANK() OVER (ORDER BY totalSales DESC) AS salesRank
    FROM
        ProductSales
    ORDER BY
        salesRank;
"""

df_product_rank = pd.read_sql(query_product_rank, conn)
print(df_product_rank)

Clasificación de Productos por Ventas Totales:
--------------------------------------------------
                             productName  totalSales  salesRank
0            1992 Ferrari 360 Spider red   239241.31          1
1                      2001 Ferrari Enzo   176710.58          2
2               1952 Alpine Renault 1300   170533.93          3
3                      1968 Ford Mustang   144121.12          4
4                       1969 Ford Falcon   143597.82          5
..                                   ...         ...        ...
104                    1982 Ducati 996 R    29020.71        105
105     1936 Mercedes Benz 500k Roadster    28618.59        106
106              1982 Lamborghini Diablo    27407.32        107
107  1958 Chevy Corvette Limited Edition    26698.98        108
108          1939 Chevrolet Deluxe Coupe    24027.56        109

[109 rows x 3 columns]


Análisis de Tendencias de Venta del '1992 Ferrari 360 Spider red'

In [ ]:
print("month-over-month sales trends for a specific product:")
print("-" * 50)

# Query to analyze month-over-month sales trends for '1992 Ferrari 360 Spider red'
query_sales_trends = """
    WITH MonthlyProductSales AS (
        SELECT
            strftime('%Y-%m', o.orderDate) AS salesMonth,
            SUM(od.quantityOrdered * od.priceEach) AS totalSales
        FROM
            orders o
        JOIN
            orderdetails od ON o.orderNumber = od.orderNumber
        JOIN
            products p ON p.productCode = od.productCode
        WHERE
            p.productName = '1992 Ferrari 360 Spider red'
        GROUP BY
            salesMonth
    )
    SELECT
        salesMonth,
        totalSales,
        -- LAG() gets the sales from the previous month
        LAG(totalSales, 1, 0) OVER (ORDER BY salesMonth) AS previousMonthSales,
        -- LEAD() gets the sales from the next month
        LEAD(totalSales, 1, 0) OVER (ORDER BY salesMonth) AS nextMonthSales
    FROM
        MonthlyProductSales
    ORDER BY
        salesMonth;
"""

df_sales_trends = pd.read_sql(query_sales_trends, conn)
print(df_sales_trends)

Análisis de Tendencias de Venta del '1992 Ferrari 360 Spider red':
--------------------------------------------------
   salesMonth  totalSales  previousMonthSales  nextMonthSales
0     2018-01     3816.85                0.00         7400.02
1     2018-03     7400.02             3816.85         8128.32
2     2018-04     8128.32             7400.02         3429.25
3     2018-05     3429.25             8128.32         3278.44
4     2018-06     3278.44             3429.25         6942.94
5     2018-07     6942.94             3278.44         4893.96
6     2018-08     4893.96             6942.94         8126.73
7     2018-09     8126.73             4893.96        13169.51
8     2018-10    13169.51             8126.73        33221.01
9     2018-11    33221.01            13169.51        11073.27
10    2018-12    11073.27            33221.01         6231.60
11    2019-01     6231.60            11073.27        12172.45
12    2019-02    12172.45             6231.60         3464.78
13    2019-03 

Clasificación de Empleados por Ventas Totales

In [ ]:
print("Rank employees:")
print("-" * 50)

# Query to rank employees by total sales
query_employee_rank = """
    WITH EmployeeSales AS (
        SELECT
            e.firstName || ' ' || e.lastName AS employeeName,
            SUM(od.quantityOrdered * od.priceEach) AS totalSales
        FROM
            employees e
        JOIN
            customers c ON e.employeeNumber = c.salesRepEmployeeNumber
        JOIN
            orders o ON c.customerNumber = o.customerNumber
        JOIN
            orderdetails od ON o.orderNumber = od.orderNumber
        GROUP BY
            employeeName
    )
    SELECT
        employeeName,
        totalSales,
        -- RANK() assigns a rank to each employee based on total sales
        RANK() OVER (ORDER BY totalSales DESC) AS salesRank
    FROM
        EmployeeSales
    ORDER BY
        salesRank;
"""

df_employee_rank = pd.read_sql(query_employee_rank, conn)
print(df_employee_rank)

Clasificación de Empleados por Ventas Totales:
--------------------------------------------------
        employeeName  totalSales  salesRank
0   Gerard Hernandez   962660.38          1
1    Leslie Jennings   884247.13          2
2    Pamela Castillo   741394.75          3
3         Larry Bott   694837.85          4
4        Barry Jones   676887.37          5
5      George Vanauf   597351.23          6
6        Loui Bondur   492709.87          7
7        Andy Fixter   479537.30          8
8     Foon Yue Tseng   459142.29          9
9         Mami Nishi   452978.58         10
10   Steve Patterson   418925.36         11
11       Peter Marsh   416923.92         12
12     Martin Gerard   387477.47         13
13    Julie Firrelli   386663.20         14
14   Leslie Thompson   347533.03         15


Análisis Completo del Producto '1992 Ferrari 360 Spider red

In [ ]:
print("Análisis Completo del Producto '1992 Ferrari 360 Spider red':")
print("-" * 50)

# -----Rango de ventas mensuales para el producto (RANK) -----
print("--- 1. Rango de ventas mensuales (RANK) ---")
query_rank = """
    SELECT
        strftime('%Y-%m', o.orderDate) AS salesMonth,
        SUM(od.quantityOrdered * od.priceEach) AS totalSales,
        RANK() OVER (ORDER BY SUM(od.quantityOrdered * od.priceEach) DESC) AS salesRank
    FROM
        orders o
    JOIN
        orderdetails od ON o.orderNumber = od.orderNumber
    JOIN
        products p ON p.productCode = od.productCode
    WHERE
        p.productName = '1992 Ferrari 360 Spider red'
    GROUP BY
        salesMonth;
"""
df_rank = pd.read_sql(query_rank, conn)
print(df_rank)
print("\n" + "="*80 + "\n")

# -----Comparación de ventas con el mes anterior y el siguiente (LAG/LEAD) -----
print("--- 2. Comparación de ventas (LAG/LEAD) ---")
query_lag_lead = """
    SELECT
        salesMonth,
        totalSales,
        LAG(totalSales, 1, 0) OVER (ORDER BY salesMonth) AS previousMonthSales,
        LEAD(totalSales, 1, 0) OVER (ORDER BY salesMonth) AS nextMonthSales
    FROM (
        SELECT
            strftime('%Y-%m', o.orderDate) AS salesMonth,
            SUM(od.quantityOrdered * od.priceEach) AS totalSales
        FROM
            orders o
        JOIN
            orderdetails od ON o.orderNumber = od.orderNumber
        JOIN
            products p ON p.productCode = od.productCode
        WHERE
            p.productName = '1992 Ferrari 360 Spider red'
        GROUP BY
            salesMonth
    ) AS monthly_sales;
"""
df_lag_lead = pd.read_sql(query_lag_lead, conn)
print(df_lag_lead)
print("\n" + "="*80 + "\n")

# ----- División de ventas en cuartiles para el rendimiento (NTILE) -----
print("--- 3. División de ventas en cuartiles (NTILE) ---")
query_ntile = """
    SELECT
        salesMonth,
        totalSales,
        NTILE(4) OVER (ORDER BY totalSales) AS salesQuartile
    FROM (
        SELECT
            strftime('%Y-%m', o.orderDate) AS salesMonth,
            SUM(od.quantityOrdered * od.priceEach) AS totalSales
        FROM
            orders o
        JOIN
            orderdetails od ON o.orderNumber = od.orderNumber
        JOIN
            products p ON p.productCode = od.productCode
        WHERE
            p.productName = '1992 Ferrari 360 Spider red'
        GROUP BY
            salesMonth
    ) AS monthly_sales;
"""
df_ntile = pd.read_sql(query_ntile, conn)
print(df_ntile)
print("\n" + "="*80 + "\n")

Análisis Completo del Producto '1992 Ferrari 360 Spider red':
--------------------------------------------------
--- 1. Rango de ventas mensuales (RANK) ---
   salesMonth  totalSales  salesRank
0     2018-11    33221.01          1
1     2019-11    21672.45          2
2     2019-10    17191.33          3
3     2019-08    15366.17          4
4     2018-10    13169.51          5
5     2019-12    12273.92          6
6     2019-02    12172.45          7
7     2018-12    11073.27          8
8     2019-06     9940.27          9
9     2020-01     9765.95         10
10    2018-04     8128.32         11
11    2018-09     8126.73         12
12    2018-03     7400.02         13
13    2019-07     7364.73         14
14    2018-07     6942.94         15
15    2019-04     6367.09         16
16    2019-01     6231.60         17
17    2020-02     5613.66         18
18    2019-05     5242.68         19
19    2018-08     4893.96         20
20    2018-01     3816.85         21
21    2019-03     3464.78    

Jerarquía de Mando para el Empleado con ID:

In [29]:
# Set the employee ID you want to start the hierarchy from
employee_id = 1056

print(f"--- Jerarquía de Mando para el Empleado con ID: {employee_id} ---")
print("-" * 50)

# Recursive query to find the complete employee hierarchy
query_recursive = f"""
    WITH RECURSIVE EmployeeHierarchy AS (
        -- Base case: Selects the initial employee to start from
        SELECT
            employeeNumber,
            reportsTo,
            firstName || ' ' || lastName AS employeeName,
            1 AS level
        FROM
            employees
        WHERE
            employeeNumber = {employee_id}

        UNION ALL

        -- Recursive case: Joins to itself to find the manager of the previous employee
        SELECT
            e.employeeNumber,
            e.reportsTo,
            e.firstName || ' ' || e.lastName AS employeeName,
            h.level + 1 AS level
        FROM
            employees e
        JOIN
            EmployeeHierarchy h ON e.employeeNumber = h.reportsTo
    )
    SELECT
        level,
        employeeName AS subordinate,
        (SELECT firstName || ' ' || lastName FROM employees WHERE employeeNumber = h.reportsTo) AS managerName
    FROM
        EmployeeHierarchy h
    ORDER BY
        level DESC;
"""

df_recursive = pd.read_sql(query_recursive, conn)
print(df_recursive)

--- Jerarquía de Mando para el Empleado con ID: 1056 ---
--------------------------------------------------
   level     subordinate   managerName
0      2    Diane Murphy          None
1      1  Mary Patterson  Diane Murphy


In [ ]:
# Close the connection
conn.close()